# RAG PIPELINE

## Install packages

In [ ]:
# Install pip first (always good practice)

!pip install --upgrade pip
# Install PyTorch (CPU version for Windows — switch to CUDA URL if you have a GPU)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu --quiet
# Install the required libraries
!pip install transformers datasets evaluate huggingface_hub pandas accelerate --quiet
!pip install ipywidgets --quiet

In [ ]:
pip freeze > requirements.txt

## Imports

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from huggingface_hub import notebook_login
import pandas as pd
from datasets import Dataset

## Hugging Face Authentication

In [ ]:


notebook_login()

## Dataset

In [ ]:


# Load AfriQA "bem" config via the HF Hub auto-converted Parquet files
splits = {}
for split in ["train", "validation", "test"]:
    url = f"https://huggingface.co/datasets/masakhane/afriqa/resolve/refs%2Fconvert%2Fparquet/bem/{split}/0000.parquet"
    splits[split] = Dataset.from_pandas(pd.read_parquet(url))

ds = splits
print(ds)
print(ds["test"][0])


## Load LLM

In [ ]:

# Define the model and tokenizer

model_name = "McGill-NLP/AfriqueQwen-8B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Simple generation test
prompt = "Je, mji mkuu wa Nigeria ni upi?"  # Swahili: "What is the capital of Nigeria?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    temperature=0.7
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)